# Chiến lược 2 bước: Pre-training & Fine-tuning
Đây là file huấn luyện mẫu dựa trên kiến trúc DAN.

## Tại sao lại cần 2 bước?
1. **Pre-training (FER2013 / AffectNet)**: Các tập dữ liệu này rất lớn, giúp mô hình học được các đặc trưng chung của khuôn mặt (mắt, mũi, miệng, góc cạnh) một cách tổng quát và chống overfitting.
2. **Fine-tuning (RAF-DB)**: RAF-DB nhỏ hơn nhưng có độ phân giải cao và sát với thực tế. Ta tải lại trọng số đã học ở Bước 1, giảm Learning Rate (để không phá hỏng các đặc trưng đã học) và tiếp tục train để mô hình hội tụ tốt hơn, cho ra Accuracy cao hơn.

In [ ]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Sử dụng thiết bị: {device}')

## 1. Định nghĩa mô hình DAN (Distract Your Attention Network)

In [ ]:
class DAN(nn.Module):
    def __init__(self, num_class=7, num_head=4):
        super(DAN, self).__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.num_head = num_head
        self.conv_att = nn.Conv2d(512, self.num_head, kernel_size=1)
        self.fc = nn.Linear(512, num_class)
        self.bn = nn.BatchNorm1d(num_class)

    def forward(self, x):
        x = self.features(x)
        att_map = self.conv_att(x)
        att_map = att_map.view(att_map.size(0), self.num_head, -1)
        att_map = F.softmax(att_map, dim=2)
        att_map = att_map.view(att_map.size(0), self.num_head, x.size(2), x.size(3))
        
        x_flat = x.view(x.size(0), 1, x.size(1), -1)
        att_flat = att_map.view(att_map.size(0), self.num_head, 1, -1)
        
        weighted_features = (x_flat * att_flat).sum(dim=-1)
        final_features = weighted_features.mean(dim=1)
        
        out = self.fc(final_features)
        out = self.bn(out)
        return out

## 2. Hàm tải dữ liệu

In [ ]:
def get_dataloaders(data_dir, batch_size=64):
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=train_transform)
    test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=test_transform)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    return train_loader, test_loader, train_dataset.classes

## 3. Hàm Huấn Luyện Chung

In [ ]:
def train_model(model, train_loader, test_loader, criterion, optimizer, scheduler, num_epochs, save_path):
    best_acc = 0.0
    for epoch in range(num_epochs):
        start_time = time.time()
        model.train()
        running_loss = 0.0
        corrects = 0
        total = 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
            total += inputs.size(0)
            
        epoch_loss = running_loss / total
        epoch_acc = corrects.double() / total
        
        model.eval()
        val_corrects = 0
        val_total = 0
        with torch.no_grad():
            for inputs, labels in test_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels.data)
                val_total += inputs.size(0)
                
        val_acc = val_corrects.double() / val_total
        scheduler.step(val_acc)
        
        epoch_time = time.time() - start_time
        print(f'Epoch {epoch+1}/{num_epochs} | Time: {epoch_time:.0f}s | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Val Acc: {val_acc:.4f}')
        
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_path)
            print(f'  -> Đã lưu model tốt nhất tại {save_path}')
            
    return best_acc

## BƯỚC 1: Pre-training trên FER2013 / AffectNet
Đường dẫn `data_dir_fer` trỏ tới tập dữ liệu lớn. Train khoảng 20-30 epochs.

In [ ]:
# Chỉ định đường dẫn tới thư mục FER2013 hoặc AffectNet
data_dir_fer = './data/FER2013' # SỬA LẠI ĐƯỜNG DẪN NÀY!

if os.path.exists(data_dir_fer):
    train_loader_fer, test_loader_fer, classes_fer = get_dataloaders(data_dir_fer, batch_size=64)
    print('Bắt đầu BƯỚC 1: Pre-training...')
    
    model_fer = DAN(num_class=len(classes_fer), num_head=4).to(device)
    criterion_fer = nn.CrossEntropyLoss()
    # Learning rate chuẩn
    optimizer_fer = optim.Adam(model_fer.parameters(), lr=0.0001, weight_decay=1e-4)
    scheduler_fer = optim.lr_scheduler.ReduceLROnPlateau(optimizer_fer, mode='max', factor=0.5, patience=3)
    
    # Bỏ comment dòng dưới để tiến hành train (Ví dụ 20 epochs)
    # train_model(model_fer, train_loader_fer, test_loader_fer, criterion_fer, optimizer_fer, scheduler_fer, num_epochs=20, save_path='pretrained_fer.pth')
else:
    print(f'Thư mục {data_dir_fer} không tồn tại. Vui lòng cập nhật đường dẫn.')

## BƯỚC 2: Fine-tuning trên RAF-DB
Sau khi có `pretrained_fer.pth`, ta khởi tạo lại mô hình, load trọng số này vào, sau đó train bằng dữ liệu RAF-DB với Learning Rate NHỎ HƠN (vd: 1e-5 thay vì 1e-4) để không phá vỡ đặc trưng cũ.

In [ ]:
# Chỉ định đường dẫn tới RAF-DB
data_dir_raf = './data/RAF-DB' # SỬA LẠI ĐƯỜNG DẪN NÀY!
pretrained_weight_path = 'pretrained_fer.pth'

if os.path.exists(data_dir_raf):
    train_loader_raf, test_loader_raf, classes_raf = get_dataloaders(data_dir_raf, batch_size=32)
    print('Bắt đầu BƯỚC 2: Fine-tuning...')
    
    # Khởi tạo mô hình mới
    model_finetune = DAN(num_class=len(classes_raf), num_head=4).to(device)
    
    # --- QUAN TRỌNG: Load trọng số từ Bước 1 ---
    if os.path.exists(pretrained_weight_path):
        model_finetune.load_state_dict(torch.load(pretrained_weight_path))
        print('Đã load trọng số pre-trained thành công!')
    else:
        print('Chưa có file pretrained_fer.pth, mô hình sẽ train từ đầu!')

    criterion_raf = nn.CrossEntropyLoss()
    # --- QUAN TRỌNG: Learning rate nhỏ đi (vd 1e-5) để fine-tune ---
    optimizer_raf = optim.Adam(model_finetune.parameters(), lr=1e-5, weight_decay=1e-4)
    scheduler_raf = optim.lr_scheduler.ReduceLROnPlateau(optimizer_raf, mode='max', factor=0.5, patience=3)
    
    # Bỏ comment dòng dưới để tiến hành train (Ví dụ 15 epochs)
    # train_model(model_finetune, train_loader_raf, test_loader_raf, criterion_raf, optimizer_raf, scheduler_raf, num_epochs=15, save_path='best_dan_raf_finetuned.pth')
else:
    print(f'Thư mục {data_dir_raf} không tồn tại. Vui lòng cập nhật đường dẫn.')